# Medical necessity payload builder

Builds trip request payloads for testing the medical necessity guidance service.

Two sources, in order of preference:

1. **Real trips from a Databricks table.** Sections 8 onward. This is the one to
   use once production read access lands.
2. **Hand written scenarios.** Sections 3 to 7. Clinical content written against
   the criteria, for cases the real data has not produced yet.

Both produce the same thing: JSON files in the Transport.net trip request shape,
with every patient identifier removed.

### The shape this builds against

Taken from `UAT_SampleTripRequest_1.json`, the sample Transport.net supplied.
Two things about it drive the design:

- `losQuestions` is **null**. Every clinical answer is in the flat `clinicalData`
  string. The structured array is not guaranteed to be populated, so payloads
  built here carry the flat form.
- The envelope has about 90 fields the service never reads, plus a 29 entry
  `additionalFields` array. They are kept so the payload is realistic in size and
  structure rather than a convenient subset.

### What the service actually reads

`tripLegs[0].clinicalData`, or `tripLegs[0].losQuestions` when populated. Nothing
else. Patient name, date of birth, SSN, MRN and insurance are not read and are
blanked here.

### What reaches a payload

| Part of the payload | Comes from |
|---|---|
| `clinicalData` | **the table.** The one column that matters |
| `olosRequestId`, `levelOfService`, `tripReason` | the table if mapped, otherwise a fallback. Echoed, never reasoned about |
| `patientInfo` | blanked. Never read from the table |
| everything else, about 90 fields | the UAT envelope, fixed |

The query is `SELECT *` so you can see what is in every column, but only the
clinical column is written into a payload. Read wide, write narrow.

### Dependencies

Standard library and what Databricks already provides. Nothing to install.

```
json, copy, shutil, os, datetime, re     standard library
spark                                     Databricks
pandas                                    via .toPandas(), in every runtime
```

Sections 1 to 9 need no cluster at all and will run on a plain Python kernel.
Only the real-data sections use `spark`.

## 1. Configuration

`OUTPUT_DIR` is a Workspace folder, so the files appear in the Databricks file
browser where they can be opened and downloaded. The leading `/Workspace` is
required: the browser URL does not show it, but it is the path a notebook sees.

Workspace files cap out around 10MB each, which these are nowhere near at roughly
13KB. If you end up generating hundreds of payloads from real trip data, switch
to a Volume instead and browse it in Catalog Explorer:

```python
OUTPUT_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/med_nec/payloads"
```

In [ ]:
import json
import copy
import shutil
import os
from datetime import datetime

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

CATALOG = "prod-sandbox"
SCHEMA = "default"
TRIP_TABLE = "temp_tnet_tripmaster"

WORKSPACE_USER = "josh.smitherman@gmr.net"

TMP_DIR = "/tmp/med_nec_payloads"
OUTPUT_DIR = f"/Workspace/Users/{WORKSPACE_USER}/med_nec/json_payloads2"

SCRUB_IDENTIFIERS = True
ROW_LIMIT = 25

os.makedirs(TMP_DIR, exist_ok=True)

print("RUN_TS      :", RUN_TS)
print("source table:", f"`{CATALOG}`.{SCHEMA}.{TRIP_TABLE}")
print("tmp dir     :", TMP_DIR)
print("output dir  :", OUTPUT_DIR)
print("scrubbing   :", SCRUB_IDENTIFIERS)
print()

if os.path.isdir(OUTPUT_DIR):
    existing = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".json")]
    print(f"output dir exists, {len(existing)} json file(s) already there")
else:
    print("output dir does not exist yet, it will be created on write")

## 2. The envelope

The real UAT payload, used as the template. Clinical content is replaced per
scenario; everything else stays as Transport.net sends it.

In [ ]:
ENVELOPE_JSON = r'''
{
 "systemId": 1,
 "draftSystemId": 1,
 "sourceSystemTripId": null,
 "msgObjType": 0,
 "msgActionType": 0,
 "correlationId": null,
 "eligibilityCheckOnly": false,
 "isTransferFacility": false,
 "isReqFacilitySearchable": true,
 "isNotifyFacilitySearchable": false,
 "isCallerFacilitySearchable": false,
 "skipEligibility": false,
 "contractName": "",
 "contractId": 229,
 "configVersionId": 137,
 "allowPastTrips": false,
 "olosRequestId": "d5c83716-0772-fe46-345c-17394bb7b828",
 "skipQueues": false,
 "pcsId": "",
 "requestQuickOrder": false,
 "requestConfigType": "",
 "requestType": 2,
 "requestTypeId": 2,
 "requestDateTime": "2026-10-01T18:17:26+00:00",
 "requesterOffset": "+00:00",
 "requesterFacilityId": 5815,
 "requesterFacility": "THR: Alliance",
 "requesterId": "9054",
 "requesterName": "QA Playwright AutomationCICD",
 "requesterFirstName": "QA Playwright",
 "requesterLastName": "AutomationCICD",
 "requesterPhone": "",
 "requestorType": "",
 "reqPrimPhone": "",
 "reqPrimPhType": "",
 "reqSecPhone": "",
 "reqSecPhType": "",
 "reqEntity": "",
 "department": "",
 "reqRole": "",
 "createdBy": "QA Playwright AutomationCICD",
 "createdByEmail": "gmrqaplaywright2@gmail.com",
 "notifyName": "Edna Quigley",
 "notifyPhone": "(324)434-3444",
 "notifyPhoneExt": "",
 "notifyEmail": "gmrqaplaywright2@gmail.com",
 "notifyFacilityId": "",
 "callerFacilityId": null,
 "preferredNotifyMethod": 1,
 "isRoundTrip": false,
 "roundTripType": 0,
 "returnTripPickupTime": "",
 "waitTimeMins": "",
 "tripComment": "",
 "tripReason": "Burn - Adult",
 "tripReasonId": "231",
 "tripReasonQuestions": null,
 "isApprovalRequired": false,
 "approvedBy": "",
 "approvedByOffset": "",
 "dateApprovedBy": "",
 "isSigningRequired": false,
 "isUserTypeSigningReq": true,
 "signedBy": "",
 "signedByName": "",
 "dateSignedBy": "",
 "signedByOffset": "",
 "signedByCredentials": "",
 "signedByLicenseNo": "",
 "signedByProfession": "",
 "signedByEmailAddress": "",
 "signedByPhoneNumber": "",
 "signedByPhoneNumberExt": "",
 "idenCovid": false,
 "userSelectedProviders": [],
 "tripLegs": [
  {
   "patientInfo": {
    "firstName": "Kendra",
    "middleName": "",
    "lastName": "Smitham",
    "dateOfBirth": "01/12/2002",
    "ssn": null,
    "mrn": "",
    "mrnSource": "",
    "visitNumber": "",
    "visitNumberSource": "",
    "gender": "Female",
    "address": "",
    "physicalStreet": "",
    "physicalCity": "",
    "physicalState": "",
    "physicalCounty": "",
    "physicalZipCode": "",
    "billingAddress": "",
    "mailingAddress": "",
    "bedAptRoom": "",
    "phoneNumber": "",
    "phoneExt": "",
    "guarantor": "",
    "diagnosis": "tester123",
    "overMaxWeight": false,
    "patientOver300Lbs": null,
    "patientWeight": "60",
    "patientHeight": "",
    "heightUnit": "0003",
    "weightUnit": "0005",
    "dischargingPhysicianNpi": "",
    "dischargingPhysician": "",
    "dischargingPhysicianPhone": "",
    "authorizingPhysician": "",
    "receivingPhysician": "",
    "attendingPhysician": "",
    "consultingPhysician": "",
    "referringPhysician": "",
    "admittingPhysician": "",
    "externalPatientId": "",
    "externalEncounterId": "",
    "ssoContractDetailsId": "",
    "epiId": ""
   },
   "insuranceInfo": {
    "planId": "",
    "payerGroup": "16",
    "planName": "",
    "insCompanyName": "",
    "payerAddress": "",
    "groupNumber": "",
    "insPolicyNumber": "",
    "addlPolicyNumber": "",
    "addlGroupNumber": "",
    "addlPlanName": "",
    "addlPlanId": "",
    "dualCoverage": null,
    "contractPlanId": "",
    "contractId": 229,
    "insOrderOfImp": 1,
    "facilityResponsible": null,
    "facilityResponsiblePlanId": null,
    "custSubscriberID": null,
    "custSourceSequence": null,
    "custSubscriberFirstName": null,
    "custSubscriberLastName": null,
    "custPlanID": null,
    "custPlanGroupName": null,
    "custPlanName": null,
    "custGroupNumber": null,
    "custIsEmbeddedPlan": false,
    "custEffectiveFromDate": null,
    "custEffectiveToDate": null,
    "custRelationshipToSubscriber": null
   },
   "additionalInsurances": [],
   "sourceSystemTripLegId": null,
   "cadId": "",
   "cadUserId": "9054",
   "msgReceived": "2026-10-01T18:17:26+00:00",
   "expiredSecs": 120,
   "preauthNumber": "",
   "referenceNumber": "",
   "authorizedBy": "",
   "addlAuthorizedBy": "",
   "addlAuthNumber": "",
   "addlAuthNumberNpi": "",
   "facilityPayOverride": false,
   "facPayOverrideReason": "",
   "facilityPayReason": null,
   "facilityPayReasonId": null,
   "systemLos": "ROTOR",
   "levelOfService": "ROTOR",
   "losCategory": "ROTOR",
   "losCategoryLabel": "Rotor",
   "losQuestions": null,
   "clinicalData": "Is this patient going for an emergent clinical intervention (Cath Lab, Neuro, Surgery)?\nIs this patient going for an emergent clinical intervention (Cath Lab, Neuro, Surgery)?\nNo\nIs this a Level One Stroke, Cardiac, or Trauma activation?\nIs this a Level One Stroke, Cardiac, or Trauma activation?\nNo\nWill another person be accompanying the patient on the flight?\nWill another person be accompanying the patient on the flight?\nNo\nRequirements for Transport\nRequirements for Transport: None - Does not apply",
   "losReason": "",
   "losOverride": false,
   "losOverrideReason": "",
   "specialNeeds": [],
   "medicareQuestionNums": [],
   "medicareQuestionTen": "DX: tester123",
   "pickupLocTypeCode": "H",
   "pickupArea": "",
   "pickupStreetNumber": "",
   "pickupAddr1": "10864 Texas Health Trl",
   "pickupBedAptRoom": "",
   "pickupBed": null,
   "pickupRoom": null,
   "pickupDepartment": "",
   "pickupFacDeptId": "",
   "pickupCity": "Keller",
   "pickupCounty": "Tarrant",
   "pickupState": "TX",
   "pickupZip": "76244",
   "pickupCountry": "",
   "pickupFacilityId": 5815,
   "pickupFacilityName": "THR: Alliance",
   "pickupFacilityMapped": true,
   "pickupAddlInfo": "",
   "pickupLightsAndSirens": null,
   "pickupLat": 32.93067,
   "pickupLong": -97.30983,
   "pickupPointId": null,
   "pickupPointLat": null,
   "pickupPointLong": null,
   "pickupOffset": "-05:00",
   "pickupTimeOffset": null,
   "pickupFacilityTypeId": "",
   "dropoffArea": "",
   "dropoffLocTypeCode": "H",
   "dropoffStreetNumber": "",
   "dropoffAddr1": "7 Medical Pkwy",
   "dropoffBedAptRoom": "",
   "dropoffBed": null,
   "dropoffRoom": null,
   "dropoffDepartment": "",
   "dropoffFacDeptId": "",
   "dropoffCity": "Dallas",
   "dropoffCounty": "Farmers Branch",
   "dropoffState": "TX",
   "dropoffZip": "75234",
   "dropoffCountry": "",
   "dropoffFacilityId": "3800",
   "dropoffFacilityName": "Dallas Medical Center",
   "dropoffFacilityMapped": true,
   "dropoffAddlInfo": "",
   "dropoffLightsAndSirens": null,
   "dropoffLat": "",
   "dropoffLong": "",
   "dropoffPointId": null,
   "dropoffPointLat": null,
   "dropoffPointLong": null,
   "dropoffMedSvc": null,
   "dropoffFacilityTypeId": 1,
   "idenOneWay": true,
   "idenReccur": false,
   "idenWillCall": false,
   "idenStandBy": false,
   "idenStandByActivated": false,
   "idenNow": true,
   "SLA": null,
   "slaTypeId": 1,
   "displayPickupTime": "10/01/2026 13:17",
   "pickupTime": "",
   "displayApptTime": "",
   "apptTime": "",
   "isApptTime": false,
   "slaId": null,
   "additionalPassengers": "",
   "transRequestComment": "",
   "tripMiles": "",
   "attachments": null,
   "preCheckMatrixId": null,
   "preCheckAuth": false,
   "preCheckBroker": false,
   "additionalFields": [
    {
     "name": "destPtReceivingName",
     "value": null,
     "dataType": "string",
     "label": "Who will be receiving patient?",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "destPtReceivingNum",
     "value": null,
     "dataType": "string",
     "label": "Receivers Phone Number",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "destPtResidenceKeys",
     "value": null,
     "dataType": "boolean",
     "label": "Does patient have keys to unlock residence?",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "destResidenceStairs",
     "value": null,
     "dataType": "boolean",
     "label": "Is the number of stairs greater than 5?",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "pUpResidenceStairs",
     "value": null,
     "dataType": "boolean",
     "label": "Is the number of stairs greater than 5?",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "ptIntubated",
     "value": null,
     "dataType": "boolean",
     "label": "Is Patient Intubated?",
     "section": "ptinfo",
     "selectedLabel": null
    },
    {
     "name": "numOfDrips",
     "value": null,
     "dataType": "string",
     "label": "Number of Drips",
     "section": "ptinfo",
     "selectedLabel": null
    },
    {
     "name": "specialEquip",
     "value": null,
     "dataType": "string",
     "label": "Special Equipment",
     "section": "ptinfo",
     "selectedLabel": null
    },
    {
     "name": "clinicalNotes",
     "value": null,
     "dataType": "string",
     "label": "Clinical Notes",
     "section": "ptinfo",
     "selectedLabel": null
    },
    {
     "name": "svcWeatherDecline",
     "value": false,
     "dataType": "boolean",
     "label": "Has any other air service declined this request due to weather?",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "svcOtherRespond",
     "value": false,
     "dataType": "boolean",
     "label": "Are any other air services responding to the transferring hospital?",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "inpatStay",
     "value": null,
     "dataType": "boolean",
     "label": "Transport following inpatient stay?",
     "section": "payerinfo",
     "selectedLabel": null
    },
    {
     "name": "pUpPtResidence",
     "value": null,
     "dataType": "boolean",
     "label": "Is pickup location member's residence?",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "destPtResidence",
     "value": null,
     "dataType": "boolean",
     "label": "Is destination location member's residence?",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "ptCovid",
     "value": null,
     "dataType": "string",
     "label": "Patient is a person under investigation or confirmed case for COVID-19",
     "section": "ptinfo",
     "selectedLabel": null
    },
    {
     "name": "admitDate",
     "value": null,
     "dataType": null,
     "label": "Admit Date",
     "section": "payerinfo",
     "selectedLabel": null
    },
    {
     "name": "groundContact",
     "value": null,
     "dataType": "string",
     "label": "Ground Contact",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "multipleUnits",
     "value": null,
     "dataType": "boolean",
     "label": "Multiple Units Requested?",
     "section": "schedinfo",
     "selectedLabel": null
    },
    {
     "name": "svcLineReq",
     "value": null,
     "dataType": "dropdown",
     "label": "Service Line Requested",
     "section": "losquestinfo",
     "selectedLabel": null
    },
    {
     "name": "locReq",
     "value": null,
     "dataType": "dropDown",
     "label": "Level of Care Requested",
     "section": "losquestinfo",
     "selectedLabel": null
    },
    {
     "name": "chiefComplaint",
     "value": null,
     "dataType": "dropDown",
     "label": "Chief Complaint",
     "section": "losquestinfo",
     "selectedLabel": null
    },
    {
     "name": "chiefComplaintDesc",
     "value": null,
     "dataType": "string",
     "label": "Chief Complaint Description",
     "section": "losquestinfo",
     "selectedLabel": null
    },
    {
     "name": "transferReason",
     "value": null,
     "dataType": "dropDown",
     "label": "Reason For Transfer",
     "section": "losquestinfo",
     "selectedLabel": null
    },
    {
     "name": "ptTransBy",
     "value": null,
     "dataType": "dropDown",
     "label": "Patient Transport By",
     "section": "losquestinfo",
     "selectedLabel": null
    },
    {
     "name": "transArrangement",
     "value": null,
     "dataType": "dropDown",
     "label": "Transport Arrangements",
     "section": "losquestinfo",
     "selectedLabel": null
    },
    {
     "name": "losClinicalNotes",
     "value": null,
     "dataType": "string",
     "label": "Clinical Summary Notes",
     "section": "losquestinfo",
     "selectedLabel": null
    },
    {
     "name": "addlRiderWeight",
     "value": null,
     "dataType": null,
     "label": "Additional Rider Weight",
     "section": "ptinfo",
     "selectedLabel": null
    },
    {
     "name": "addlRiderName",
     "value": null,
     "dataType": null,
     "label": "Additional Rider Name",
     "section": "losquestinfo",
     "selectedLabel": null
    },
    {
     "name": "addlRiderDob",
     "value": null,
     "dataType": null,
     "label": "Additional Rider DOB",
     "section": "losquestinfo",
     "selectedLabel": null
    }
   ],
   "transportDuration": 38,
   "transportDurationTraffic": null,
   "transportDistance": 33,
   "tripConcierge": {
    "isTransportEligible": false,
    "isAvailable": false,
    "unavailableReasonId": 2
   }
  }
 ]
}
'''

ENVELOPE = json.loads(ENVELOPE_JSON)

print("top level fields :", len(ENVELOPE))
print("leg fields       :", len(ENVELOPE["tripLegs"][0]))
print("additionalFields :", len(ENVELOPE["tripLegs"][0]["additionalFields"]))
print("losQuestions     :", ENVELOPE["tripLegs"][0]["losQuestions"])
print("levelOfService   :", ENVELOPE["tripLegs"][0]["levelOfService"])

## 3. The blank patient block

Every field emptied. Kept in the payload rather than deleted so the structure
still matches what Transport.net sends, and so the absence is visible rather
than silent.

In [ ]:
PATIENT_TEMPLATE = ENVELOPE["tripLegs"][0]["patientInfo"]

BLANK_PATIENT = {}
for key in PATIENT_TEMPLATE:
    value = PATIENT_TEMPLATE[key]
    if isinstance(value, bool):
        BLANK_PATIENT[key] = False
    elif value is None:
        BLANK_PATIENT[key] = None
    else:
        BLANK_PATIENT[key] = ""

BLANK_PATIENT["heightUnit"] = "0003"
BLANK_PATIENT["weightUnit"] = "0005"
BLANK_PATIENT["_note"] = "Deliberately empty. The service reads only clinical answers."

print(f"{len(BLANK_PATIENT)} fields blanked")
for key in list(BLANK_PATIENT)[:8]:
    print(f"  {key:22} {BLANK_PATIENT[key]!r}")
print("  ...")

## 4. Scenario definitions

Each entry is one trip. `items` is a list of question and answer pairs; they get
flattened into `clinicalData` in the next cell, in the format Transport.net uses.

Edit this list to add cases. Nothing below needs changing.

In [ ]:
SCENARIOS = [
    {
        "stem": "uat_2_vent_bed_confined",
        "olosRequestId": "e1a00001-0000-4000-8000-000000000002",
        "tripReason": "Discharge to Skilled Nursing Facility",
        "tripReasonId": "118",
        "los": "ALS",
        "dropoffLocTypeCode": "H",
        "medicareQuestionTen": "DX: Respiratory failure, ventilator dependent",
        "expected": "meets",
        "items": [
            ("Is this patient being transferred to a higher level of care due to their clinical condition?",
             "No (Discharge, Behavioral Transfer or Other Reason): dc to SNF"),
            ("Is any special handling required during transport?",
             "The patient is bed confined before and after transport. All three criteria must be met in order to use bed confined. Please specify reason in comments field.: The patient is unable to get up from bed without assistance, The patient is unable to ambulate, The patient is unable to sit in a chair or wheelchair: vent dependent, trach in place"),
            ("Does the patient require airway support during transport?",
             "Patient is on a ventilator operated by the transport provider, Requires regular oropharyngeal suctioning"),
            ("Is this a Time Sensitive transport request?", "No"),
        ],
    },
    {
        "stem": "uat_3_general_weakness",
        "olosRequestId": "e1a00001-0000-4000-8000-000000000003",
        "tripReason": "Discharge to Residence",
        "tripReasonId": "121",
        "los": "BLS",
        "dropoffLocTypeCode": "R",
        "medicareQuestionTen": "DX: Deconditioning",
        "expected": "does_not_meet, asks, flips to meets",
        "items": [
            ("Is this patient being transferred to a higher level of care due to their clinical condition?",
             "No (Discharge, Behavioral Transfer or Other Reason): discharge home"),
            ("Is any special handling required during transport?",
             "The patient is unable to sit safely in a wheelchair while the vehicle is in motion"),
            ("Please specify the reason the patient cannot sit safely in a wheelchair",
             "general weakness"),
            ("Does the patient require airway support during transport?", "None - Does not apply"),
            ("Is this a Time Sensitive transport request?", "No"),
        ],
    },
    {
        "stem": "uat_4_wheelchair_candidate",
        "olosRequestId": "e1a00001-0000-4000-8000-000000000004",
        "tripReason": "Transport to Scheduled Appointment",
        "tripReasonId": "140",
        "los": "BLS",
        "dropoffLocTypeCode": "H",
        "medicareQuestionTen": "DX: End stage renal disease",
        "expected": "does_not_meet, asks, stays does_not_meet",
        "items": [
            ("Is this patient being transferred to a higher level of care due to their clinical condition?",
             "No (Discharge, Behavioral Transfer or Other Reason): to dialysis"),
            ("Is any special handling required during transport?",
             "The patient can sit safely in a wheelchair while the vehicle is in motion"),
            ("Does the patient require airway support during transport?",
             "The patient requires continuous oxygen provided by the patient's own equipment at their baseline rate"),
            ("Is this a Time Sensitive transport request?",
             "Yes - Due to other reasons such as destination closing soon: dialysis chair time at 1400"),
        ],
    },
]

print(f"{len(SCENARIOS)} scenarios")
for s in SCENARIOS:
    print(f"  {s['stem']:30} {s['los']:6} {len(s['items'])} items   expect: {s['expected']}")

## 5. Flatten to clinicalData

Transport.net writes each item as the question twice, then the answer:

```
Is this a Time Sensitive transport request?
Is this a Time Sensitive transport request?
No
```

or, where the label and the question are the same thing, as `label` then
`label: answer`. This reproduces the first form, which is what the ground
examples in the UAT sample use.

In [ ]:
FLATTENED = {}

for s in SCENARIOS:
    lines = []
    for question, answer in s["items"]:
        lines.append(question)
        lines.append(question)
        lines.append(answer)
    FLATTENED[s["stem"]] = "\n".join(lines)

for stem in FLATTENED:
    blob = FLATTENED[stem]
    print("=" * 78)
    print(stem, f"({len(blob.splitlines())} lines, {len(blob)} chars)")
    print("=" * 78)
    for line in blob.splitlines()[:6]:
        print("  ", line[:110])
    print("   ...")
    print()

## 6. Assemble the payloads

Deep copy of the envelope per scenario, clinical content swapped in,
`losQuestions` left null to match the real payload, requester details blanked.

In [ ]:
PAYLOADS = {}

for s in SCENARIOS:
    trip = copy.deepcopy(ENVELOPE)
    leg = trip["tripLegs"][0]

    trip["olosRequestId"] = s["olosRequestId"]
    trip["tripReason"] = s["tripReason"]
    trip["tripReasonId"] = s["tripReasonId"]
    trip["requesterName"] = "UAT Automation"
    trip["requesterFirstName"] = "UAT"
    trip["requesterLastName"] = "Automation"
    trip["createdBy"] = "UAT Automation"
    trip["createdByEmail"] = ""
    trip["notifyName"] = ""
    trip["notifyEmail"] = ""
    trip["notifyPhone"] = ""

    leg["patientInfo"] = copy.deepcopy(BLANK_PATIENT)
    leg["systemLos"] = s["los"]
    leg["levelOfService"] = s["los"]
    leg["losCategory"] = s["los"]
    leg["losCategoryLabel"] = s["los"]
    leg["losQuestions"] = None
    leg["clinicalData"] = FLATTENED[s["stem"]]
    leg["medicareQuestionTen"] = s["medicareQuestionTen"]
    leg["dropoffLocTypeCode"] = s["dropoffLocTypeCode"]

    PAYLOADS[s["stem"]] = trip

for stem in PAYLOADS:
    leg = PAYLOADS[stem]["tripLegs"][0]
    print(f"{stem:30} {leg['levelOfService']:6} losQuestions={leg['losQuestions']}  "
          f"clinicalData={len(leg['clinicalData'])} chars")

## 7. Verify before writing

Nothing leaves this notebook until it passes. The check looks for any value left
in `patientInfo`, and for the specific identifiers that were in the UAT sample.

In [ ]:
LEAK_TOKENS = ["Kendra", "Smitham", "01/12/2002", "gmrqaplaywright",
               "Edna", "Quigley", "tester123"]

ALLOWED_PATIENT_KEYS = ["heightUnit", "weightUnit", "_note"]

failures = []

for stem in PAYLOADS:
    trip = PAYLOADS[stem]
    blob = json.dumps(trip)

    patient = trip["tripLegs"][0]["patientInfo"]
    populated = []
    for key in patient:
        if key in ALLOWED_PATIENT_KEYS:
            continue
        if patient[key] not in ("", None, False):
            populated.append(key)

    hits = []
    for token in LEAK_TOKENS:
        if token.lower() in blob.lower():
            hits.append(token)

    if populated or hits:
        failures.append((stem, populated, hits))
        print(f"FAIL  {stem}")
        if populated:
            print(f"        patientInfo still populated: {populated}")
        if hits:
            print(f"        identifier tokens found: {hits}")
    else:
        print(f"PASS  {stem}")

print()
if failures:
    raise ValueError(f"{len(failures)} payload(s) failed the identifier check. Nothing written.")
print(f"All {len(PAYLOADS)} payloads clean.")

## 8. Write them out

`/tmp` first, then copied with `shutil.copyfile`, which is the pattern that works
on serverless. If the copy fails the files stay in `/tmp` and the paths are
printed rather than the run failing silently.

In [ ]:
written = []

for stem in PAYLOADS:
    tmp_path = f"{TMP_DIR}/{stem}_{RUN_TS}.json"
    with open(tmp_path, "w") as handle:
        json.dump(PAYLOADS[stem], handle, indent=4)
    written.append((stem, tmp_path))

print(f"wrote {len(written)} files to {TMP_DIR}")
for stem, path in written:
    print(f"  {os.path.getsize(path):>8,} bytes  {path}")

In [ ]:
copied = []
failed_copy = []

for stem, tmp_path in written:
    target = f"{OUTPUT_DIR}/{os.path.basename(tmp_path)}"
    try:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        shutil.copyfile(tmp_path, target)
        copied.append(target)
    except Exception as exc:
        failed_copy.append((stem, f"{type(exc).__name__}: {exc}"))

if copied:
    print(f"copied {len(copied)} files to {OUTPUT_DIR}")
    for path in copied:
        print("  ", path)
    print()
    print("Open them in the Databricks file browser at:")
    print(f"  Workspace > Users > {WORKSPACE_USER} > med_nec > json_payloads2")

if failed_copy:
    print()
    print(f"{len(failed_copy)} could not be copied. They are still in {TMP_DIR}.")
    for stem, err in failed_copy:
        print(f"  {stem}: {err[:120]}")
    print()
    print("Check that OUTPUT_DIR starts with /Workspace and that the path exists.")
    print("If this is a large run, use a Volume instead:")
    print(f"  /Volumes/{CATALOG}/{SCHEMA}/med_nec/payloads")

## 9. Print one payload

To check the shape, or to copy a single payload out by hand.

In [ ]:
STEM_TO_SHOW = "uat_3_general_weakness"

trip = PAYLOADS[STEM_TO_SHOW]
leg = trip["tripLegs"][0]

print("olosRequestId  :", trip["olosRequestId"])
print("tripReason     :", trip["tripReason"])
print("levelOfService :", leg["levelOfService"])
print("losQuestions   :", leg["losQuestions"])
print()
print("clinicalData:")
print("-" * 78)
print(leg["clinicalData"])
print("-" * 78)

In [ ]:
print(json.dumps(PAYLOADS[STEM_TO_SHOW], indent=2)[:3000])
print("...")

## 10. Building from real trip data

Everything above is hand written clinical content. This section builds the same
payloads from actual trips once read access lands.

The table and column names below are a starting point, not a verified mapping.
Run the next two cells first and set `COLUMN_MAP` from what they show.

In [ ]:
tables = spark.sql(f"SHOW TABLES IN `{CATALOG}`.{SCHEMA}")
display(tables)

In [ ]:
schema_df = spark.sql(f"DESCRIBE `{CATALOG}`.{SCHEMA}.{TRIP_TABLE}")
display(schema_df)

Pull everything, then filter on the DataFrame. Narrowing at query time hides what
is in the columns you did not ask for, which is how the `TransferReason` problem
went unnoticed for a full scoring run.

In [ ]:
raw = spark.sql(f"SELECT * FROM `{CATALOG}`.{SCHEMA}.{TRIP_TABLE} LIMIT {ROW_LIMIT}").toPandas()

print("rows     :", len(raw))
print("columns  :", len(raw.columns))
print()
for col in sorted(raw.columns):
    non_null = raw[col].notna().sum()
    distinct = raw[col].nunique(dropna=True)
    print(f"  {col:38} non-null {non_null:>4}   distinct {distinct:>4}")

### One required column

`clinical_data` is the only column that is read into the payload and the only one
the service evaluates. Set it and the notebook works.

The three below it are optional. They are metadata the service echoes back and
never reasons about, and each has a fallback:

| Role | If not mapped |
|---|---|
| `olos_request_id` | a generated id, `generated-0000` |
| `level_of_service` | `BLS` |
| `trip_reason` | left empty |

`olos_request_id` is the one worth mapping anyway, because without it a payload
cannot be traced back to the trip it came from when a reviewer disagrees with a
determination. `level_of_service` is worth mapping if the table carries air
transports, since those are out of scope and better screened out here than sent
to the model.

Everything else in the table stays in the DataFrame and never reaches a payload.

In [ ]:
CLINICAL_DATA_COLUMN = "ClinicalData"

OPTIONAL_COLUMNS = {
    "olos_request_id": "OlosRequestId",
    "level_of_service": "LevelOfService",
    "trip_reason": "TripReason",
}

if CLINICAL_DATA_COLUMN not in raw.columns:
    print(f"FAIL   '{CLINICAL_DATA_COLUMN}' is not in the table.")
    print("       Set CLINICAL_DATA_COLUMN from the column list above.")
    raise ValueError(f"required column not found: {CLINICAL_DATA_COLUMN}")

print(f"PASS   clinical data column: {CLINICAL_DATA_COLUMN}")
print()

RESOLVED = {}
for role in OPTIONAL_COLUMNS:
    column = OPTIONAL_COLUMNS[role]
    if column in raw.columns:
        RESOLVED[role] = column
        print(f"       {role:20} -> {column}")
    else:
        RESOLVED[role] = None
        print(f"       {role:20} -> not mapped, fallback will be used")

A trip is only usable if `clinicalData` actually holds clinical text. Rows where
it is empty, or where it is a short status value rather than question and answer
content, are dropped here rather than sent to the model.

That guard exists because of a real failure: `TransferReason` turned out to be a
workflow status field with a handful of distinct values, and 500 model calls were
spent on it before anyone noticed.

In [ ]:
clinical_col = CLINICAL_DATA_COLUMN

usable = raw[raw[clinical_col].notna()].copy()
usable["clinical_len"] = usable[clinical_col].astype(str).str.len()
usable["line_count"] = usable[clinical_col].astype(str).str.count("\n") + 1

distinct_values = usable[clinical_col].nunique(dropna=True)
median_lines = int(usable["line_count"].median()) if len(usable) else 0

print(f"rows with clinical data : {len(usable)} of {len(raw)}")
print(f"distinct values         : {distinct_values}")
print(f"median length           : {int(usable['clinical_len'].median()) if len(usable) else 0}")
print(f"median line count       : {median_lines}")
print()

if len(usable) == 0:
    print("FAIL   No rows have clinical data. Wrong column, or ROW_LIMIT is too small.")

elif median_lines < 3:
    print("CHECK  Values are mostly single lines. Expected question and answer blocks")
    print("       across several lines. This may be a status or category field.")
    print()
    for value in usable[clinical_col].astype(str).head(5):
        print("  ", value[:140])

elif len(usable) >= 30 and distinct_values <= 60:
    print("CHECK  Many rows but few distinct values. This looks like a status field")
    print("       rather than per-trip clinical text. Confirm before building.")
    print()
    print(usable[clinical_col].value_counts().head(15).to_string())

elif len(usable) < 30:
    print("PASS   Content looks like question and answer blocks.")
    print(f"       Note: {len(usable)} rows is too small to judge whether the column")
    print("       repeats across trips. Raise ROW_LIMIT before trusting that signal.")

else:
    print("PASS   Looks like per-trip question and answer content.")

In [ ]:
REAL_PAYLOADS = {}

for position in range(len(usable)):
    row = usable.iloc[position]

    trip = copy.deepcopy(ENVELOPE)
    leg = trip["tripLegs"][0]

    if RESOLVED["olos_request_id"]:
        request_id = str(row[RESOLVED["olos_request_id"]])
    else:
        request_id = f"generated-{position:04d}"

    if RESOLVED["level_of_service"]:
        los = str(row[RESOLVED["level_of_service"]])
    else:
        los = "BLS"

    if RESOLVED["trip_reason"]:
        trip_reason = str(row[RESOLVED["trip_reason"]])
    else:
        trip_reason = ""

    trip["olosRequestId"] = request_id
    trip["tripReason"] = trip_reason
    trip["requesterName"] = ""
    trip["requesterFirstName"] = ""
    trip["requesterLastName"] = ""
    trip["createdBy"] = ""
    trip["createdByEmail"] = ""
    trip["notifyName"] = ""
    trip["notifyEmail"] = ""
    trip["notifyPhone"] = ""

    leg["patientInfo"] = copy.deepcopy(BLANK_PATIENT)
    leg["systemLos"] = los
    leg["levelOfService"] = los
    leg["losCategory"] = los
    leg["losCategoryLabel"] = los
    leg["losQuestions"] = None
    leg["clinicalData"] = str(row[clinical_col])
    leg["medicareQuestionTen"] = ""

    REAL_PAYLOADS[f"real_{position:03d}_{request_id[:8]}"] = trip

print(f"built {len(REAL_PAYLOADS)} payloads from real trips")
for stem in list(REAL_PAYLOADS)[:10]:
    leg = REAL_PAYLOADS[stem]["tripLegs"][0]
    print(f"  {stem:28} {leg['levelOfService']:8} {len(leg['clinicalData']):>6} chars")

Same identifier check as section 7, run against the real payloads. Free text
written by a requester can name a patient even when the structured fields are
clean, so this also looks for anything that reads like a name or a date of birth
in `clinicalData`.

In [ ]:
import re

DOB_PATTERN = re.compile(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b")
SSN_PATTERN = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
MRN_PATTERN = re.compile(r"\b(?:mrn|medical record)\s*[:#]?\s*\w+", re.IGNORECASE)

real_failures = []

for stem in REAL_PAYLOADS:
    trip = REAL_PAYLOADS[stem]
    patient = trip["tripLegs"][0]["patientInfo"]
    clinical = trip["tripLegs"][0]["clinicalData"]

    populated = []
    for key in patient:
        if key in ALLOWED_PATIENT_KEYS:
            continue
        if patient[key] not in ("", None, False):
            populated.append(key)

    dob_hits = DOB_PATTERN.findall(clinical)
    ssn_hits = SSN_PATTERN.findall(clinical)
    mrn_hits = MRN_PATTERN.findall(clinical)

    if populated or dob_hits or ssn_hits or mrn_hits:
        real_failures.append(stem)
        print(f"CHECK  {stem}")
        if populated:
            print(f"         patientInfo populated: {populated}")
        if dob_hits:
            print(f"         date-like strings in clinicalData: {dob_hits[:3]}")
        if ssn_hits:
            print(f"         SSN-like strings: {len(ssn_hits)} found")
        if mrn_hits:
            print(f"         MRN-like strings: {mrn_hits[:2]}")

print()
if real_failures:
    print(f"{len(real_failures)} of {len(REAL_PAYLOADS)} need review before they leave Databricks.")
    print("Free text can carry identifiers the structured fields do not. Read them.")
else:
    print(f"All {len(REAL_PAYLOADS)} clean on the automated checks.")
    print("Automated checks are not a substitute for reading a sample.")

In [ ]:
real_written = []

for stem in REAL_PAYLOADS:
    if stem in real_failures:
        continue
    tmp_path = f"{TMP_DIR}/{stem}_{RUN_TS}.json"
    with open(tmp_path, "w") as handle:
        json.dump(REAL_PAYLOADS[stem], handle, indent=4)
    real_written.append((stem, tmp_path))

print(f"wrote {len(real_written)} of {len(REAL_PAYLOADS)}")
if real_failures:
    print(f"held back {len(real_failures)} that did not pass the identifier check")
for stem, path in real_written[:10]:
    print("  ", path)

## What to do with these

Drop the JSON files into `med_nec_api/payloads/` and the service picks them up on
restart. Each one needs a paired response in `responses/` with the same stem, or
the loader refuses to start.

For the hand written scenarios those responses already exist. For payloads built
from real trips there is no expected answer yet, which is the point: those are
what the model gets measured against once a reviewer has labelled them.